In [1]:
platformID = 'FBE'


In [2]:
from datetime import datetime
import pandas as pd

import os 

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from functions import lookup_loader, execute_sql_query, calculate_rolling_avg_country_split, apply_first_split_backfill, compare_or_update_reference, fix_country_custom_pct_rel
import test_functions

from config import gam_info

In [4]:
lookup = lookup_loader(gam_info, platformID, '1c',
                       with_country=True, country_col=['YT-_FBE_codes'])
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


✅ Test FBE_1c_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_02 passed: No empty values in lookup.
...updating logbook...

✅ Test FBE_1c_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_05 passed: No empty values in lookup.
...updating logbook...

✅ Test FBE_1c_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_08 passed: No empty values in lookup.
...updating logbook...



## country

In [5]:
sql_query = f"""
    SELECT 
        week_commencing,
        page_id ,
        page_name,
        page_fans_country_total AS page_fans_country,
        country_code AS fb_metric_breakdown
    FROM
        redshiftdb.central_insights.adverity_social_facebook_page_fans_by_country
    WHERE
        week_commencing Between '{gam_info['w/c_start']}' and '{gam_info['w/c_end']}'
        ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = platformID+facebook_country_raw['page_id']
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

facebook_country_raw = pd.read_csv(file, keep_default_na=False, dtype={"page_id": "string"}).drop_duplicates()

if not facebook_country_raw['page_id'].str.startswith(platformID, na=False).all():
    facebook_country_raw['page_id'] = platformID + facebook_country_raw['page_id']

facebook_country_raw['week_commencing'] = pd.to_datetime(facebook_country_raw['week_commencing'])
facebook_country_raw = facebook_country_raw.rename(columns={'page_id': 'Channel ID',
                                                            'page_name': 'Channel Name',
                                                            'week_commencing': 'w/c',
                                                            'fb_metric_breakdown': 'YT-_FBE_codes'})

no redshift connection using last time queried


In [6]:
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()

### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(facebook_country_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_09",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=facebook_country_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(facebook_country_raw, 
                           numeric_columns=['page_fans_country'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(facebook_country_raw, ['Channel ID', 'w/c', 'YT-_FBE_codes'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
✅ Test FBE_1c_09 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
               Channel ID      Start End        w/c
3827   FBE630866223444617 2025-06-23 NaT 2025-06-23
1288   FBE151173098230967 2020-04-01 NaT 2025-06-23
3828   FBE630866223444617 2025-06-23 NaT 2025-06-30
14     FBE100163990128209 2020-04-01 NaT 2025-07-07
1861   FBE173190249410689 2020-04-01 NaT 2025-07-07
...                   ...        ...  ..        ...
2858      FBE285361880228 2020-04-01 NaT 2026-01-26
1143  FBE1385243528221504 2020-04-01 NaT 2026-01-26
1846   FBE171824429536304 2025-04-02 NaT 2026-01-26
3990   FBE660673490805047 2020-04-01 NaT 2026-01-26
4474   FBE948946275170651 2020-04-01 NaT 2026-01-26

[831 rows x 4 columns]

Summary of missing weeks per channel_id_col:
             Channel ID  missing_week_count
43   FBE171824429536304                  26
17   FBE124158667615790                  14
79   FBE359687864111179                  13
69 

In [7]:
# filter to relevant channel ids
facebook_country = facebook_country_raw[facebook_country_raw['Channel ID'].isin(channel_ids)]

# fill missing countries 
facebook_country['YT-_FBE_codes'] = facebook_country['YT-_FBE_codes'].replace('', 'ZZ')

# filter to relevant countries
test_functions.test_inner_join(facebook_country, 
                               country_codes, 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_13",
                               test_step='calculating country %',
                                focus='left')

facebook_country = facebook_country.merge(country_codes[['YT-_FBE_codes', 'PlaceID']], 
                                          on='YT-_FBE_codes', 
                                          how='left').drop(columns=['YT-_FBE_codes'])


✅ Inner join test FBE_1c_13 successful: No issues found.
...updating logbook...



/var/folders/nk/x7c25nz92c1d6ljxz7l891cm0000gn/T/ipykernel_53596/2516926950.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  facebook_country['YT-_FBE_codes'] = facebook_country['YT-_FBE_codes'].replace('', 'ZZ')


In [8]:

# Group by specified columns and sum the fb_metric_value
facebook_country_sum = facebook_country.groupby(['Channel ID', 'w/c'])\
                                        .agg(Sum_page_fans_country=('page_fans_country', 'sum'))\
                                        .reset_index()
facebook_country = facebook_country.merge(facebook_country_sum, 
                                          how='inner', on= ['Channel ID', 'w/c'])
test_functions.test_inner_join(facebook_country, facebook_country_sum, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_14", 
                               test_step='calculating country %')

facebook_country['country_%'] = facebook_country['page_fans_country']/facebook_country['Sum_page_fans_country']

### RUN TESTS
test_functions.test_percentage(facebook_country, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_15", 
                               test_step='summing country % per week & account')

✅ Inner join test FBE_1c_14 successful: No issues found.
...updating logbook...

...updating logbook...



In [9]:
# calculate rolling average for missing weeks ?
avg_country_df = calculate_rolling_avg_country_split(facebook_country, 'country_%', 
                                                     week_tester['w/c'].min(), week_tester['w/c'].max())

# new channels have missing country splits for the first few weeks
avg_backfill_country_df = apply_first_split_backfill(avg_country_df, 
                                                          socialmedia_accounts, 
                                                          week_tester
                                                         )


In [ ]:
# calculate rolling average for missing weeks ?
avg_country_df = calculate_rolling_avg_country_split(facebook_country, 'country_%', 
                                                     week_tester['w/c'].min(), week_tester['w/c'].max())

# new channels have missing country splits for the first few weeks
avg_backfill_country_df = apply_first_split_backfill(avg_country_df, 
                                                          socialmedia_accounts, 
                                                          week_tester
                                                         )


In [11]:
    
# Iran = PlaceID "IRN"
IRAN_CODE = "IRN"

result_df = fix_country_custom_pct_rel(
    result_df,
    IRAN_CODE,
    {
    "2026-01-12": 0.01,
    "2026-01-19": 0.67
}
)

Fixing country percentages with mapping (multipliers):
{'2026-01-12': 0.01, '2026-01-19': 0.67}
Affected rows: (9805, 5)


/Users/dominiquebruns/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:193: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  affected = df[df['w/c'].isin(affected_weeks)]


In [ ]:
#sanity checks 
# 1
#display(result_df[(result_df['w/c'] == '2026-01-12') & (result_df['PlaceID'] == 'IRN')].head())
# 2
#display(result_df[(result_df['w/c'] == '2026-01-12') & (result_df['Channel ID'] == 'FBE102186576502930')].sort_values('PlaceID'))
# 3 
print(result_df[(result_df['w/c'] == '2026-01-12') & (result_df['Channel ID'] == 'FBE102186576502930')].sort_values('PlaceID')['country_%'].sum())

#sanity checks - previous weeks are not affected
# 4
#display(result_df[(result_df['w/c'] == '2026-01-05') & (result_df['PlaceID'] == 'IRN')].head())
# 5
#display(result_df[(result_df['w/c'] == '2026-01-05') & (result_df['Channel ID'] == 'FBE102186576502930')].sort_values('PlaceID'))
# 6 
print(result_df[(result_df['w/c'] == '2026-01-05') & (result_df['Channel ID'] == 'FBE102186576502930')].sort_values('PlaceID')['country_%'].sum())

1.0023698720678935
1.001293690488409


In [13]:
# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=result_df,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_16",
                                               test_step="After rolling average: Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(result_df, 
                           numeric_columns=['country_%'], 
                           test_number=f"{platformID}_1c_17",
                           test_step='After rolling average: Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(result_df, 
                               ['Channel ID', 'w/c', 'PlaceID'], 
                               test_number=f"{platformID}_1c_18",
                               test_step='After rolling average: Check no duplicates from redshift returned')

✅ No missing weeks from each channel’s Start onward.
...updating logbook...

✅ Test FBE_1c_17 passed: No NaN and all numeric values > 0.
...updating logbook...

✅ Test FBE_1c_18 passed: No combinations occurs more than once a week.
...updating logbook...



In [14]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
result_df[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_COUNTRY.csv",
                        index=None)